# Conduct a analysis of the performance


In [42]:
import pandas
import duckdb
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import seaborn as sns

# 1.Read & load the results from the databases crash test !

In [9]:
file_path = 'benchmarker/database_benchmark_results.csv'
results_df = pandas.read_csv(file_path)
results_df.columns

Index(['query', 'original_query', 'database_type', 'execution_time_ms',
       'cpu_usage_percent', 'memory_usage_mb', 'memory_usage_percent',
       'disk_read_mb', 'disk_write_mb', 'network_in_mb', 'network_out_mb',
       'result_rows', 'result_size_mb', 'failed'],
      dtype='object')

In [10]:
# check if results return failed queries
failed = results_df[results_df['failed'] == True]
failed

#print(failed['original_query'].values)

,query,original_query,database_type,execution_time_ms,cpu_usage_percent,memory_usage_mb,memory_usage_percent,disk_read_mb,disk_write_mb,network_in_mb,network_out_mb,result_rows,result_size_mb,failed
42,/* The INTERVAL creates an error in mysql */ /...,-- The INTERVAL creates an error in mysql\n\n-...,ClickHouseHandler,61.998129,6.2551,836.117188,20.413017,0.0,0.0,0.002766,0.004555,0,0.0,True


In [11]:
# Display initial data info
results_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44 entries, 0 to 43
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   query                 44 non-null     object 
 1   original_query        44 non-null     object 
 2   database_type         44 non-null     object 
 3   execution_time_ms     44 non-null     float64
 4   cpu_usage_percent     44 non-null     float64
 5   memory_usage_mb       44 non-null     float64
 6   memory_usage_percent  44 non-null     float64
 7   disk_read_mb          44 non-null     float64
 8   disk_write_mb         44 non-null     float64
 9   network_in_mb         44 non-null     float64
 10  network_out_mb        44 non-null     float64
 11  result_rows           44 non-null     int64  
 12  result_size_mb        44 non-null     float64
 13  failed                44 non-null     bool   
dtypes: bool(1), float64(9), int64(1), object(3)
memory usage: 4.6+ KB


In [12]:
results_df.describe()

,execution_time_ms,cpu_usage_percent,memory_usage_mb,memory_usage_percent,disk_read_mb,disk_write_mb,network_in_mb,network_out_mb,result_rows,result_size_mb
count,44.000000,44.000000,44.000000,44.000000,44.0,44.0,44.000000,44.000000,44.000000,44.000000
mean,280.579908,3.548964,469.718306,15.125782,0.0,0.0,0.008700,1.361147,42816.818182,1.236386
std,424.818061,12.430645,291.635162,3.919215,0.0,0.0,0.018827,2.849215,72302.088789,2.008356
min,3.624916,0.002186,91.125000,8.898926,0.0,0.0,0.000000,0.000000,0.000000,0.000000
25%,38.366973,0.035220,172.292969,12.556839,0.0,0.0,0.001117,0.001268,1.000000,0.000149
50%,125.228047,0.599989,459.949219,15.779781,0.0,0.0,0.002076,0.004548,100.000000,0.006992
75%,214.842975,3.811686,749.439453,18.296862,0.0,0.0,0.003363,0.369423,48204.000000,1.404179
max,1680.254936,82.681051,910.074219,22.218609,0.0,0.0,0.077224,9.390633,166536.000000,5.082401


# 2. Analyse the results with SQL in duckdb

In [13]:
results_db = duckdb.sql("SELECT * FROM results_df")
type(results_db)

duckdb.duckdb.DuckDBPyRelation

In [14]:
results_df.columns 

Index(['query', 'original_query', 'database_type', 'execution_time_ms',
       'cpu_usage_percent', 'memory_usage_mb', 'memory_usage_percent',
       'disk_read_mb', 'disk_write_mb', 'network_in_mb', 'network_out_mb',
       'result_rows', 'result_size_mb', 'failed'],
      dtype='object')

## Overall Performance Analysis

## Execution time

In [32]:
# get average, median, min, max execution time
execution_time = duckdb.sql("""
SELECT 
    database_type,
    ROUND(AVG(execution_time_ms), 2) as avg_execution_time_ms,
    ROUND(MEDIAN(execution_time_ms), 2) as median_execution_time_ms,
    ROUND(MIN(execution_time_ms), 2) as min_execution_time_ms, 
    ROUND(MAX(execution_time_ms), 2) as max_execution_time_ms,
    ROUND(SUM(execution_time_ms),2) as total_execution_time
FROM results_df
GROUP BY database_type
ORDER BY avg_execution_time_ms;
""")

execution_time

┌───────────────────┬───────────────────────┬──────────────────────────┬───────────────────────┬───────────────────────┬──────────────────────┐
│   database_type   │ avg_execution_time_ms │ median_execution_time_ms │ min_execution_time_ms │ max_execution_time_ms │ total_execution_time │
│      varchar      │        double         │          double          │        double         │        double         │        double        │
├───────────────────┼───────────────────────┼──────────────────────────┼───────────────────────┼───────────────────────┼──────────────────────┤
│ DuckDBHandler     │                 71.17 │                    13.28 │                  3.62 │                242.34 │                782.9 │
│ ClickHouseHandler │                276.81 │                    63.32 │                 31.23 │               1229.81 │              3044.89 │
│ PostgresHandler   │                350.82 │                   160.51 │                119.89 │               1680.25 │              38

In [ ]:
'''# Find the median CPU usage percent
median_cpu_usage = results_df['cpu_usage_percent'].median()

# Filter the DataFrame to include only rows where cpu_usage_percent equals the median
results_df_cpu = results_df[results_df['cpu_usage_percent'] == median_cpu_usage]

# Pivot the DataFrame to create the heatmap data
heatmap_data = results_df_cpu.pivot(
    index='query',
    columns='database_type',
    values='cpu_usage_percent'
)

# Create the heatmap using Plotly Express
fig = px.imshow(
    heatmap_data,
    title='CPU Utilization Percentage Heatmap (Query vs Database)',
    #labels={'database_type': 'Database System', 'query': 'Query', 'cpu_usage_percent': 'Average CPU Utilization (%)'}
)
fig.update_layout(
    xaxis=dict(side="top"),  # Place database names at the top
    height=800  # Adjust height based on number of queries
)
fig.show()
'''

### Total & Median execution time by Database

In [135]:
# define the template for all charts
template='plotly_dark'

execution_time_df = execution_time.df()

total_execution_plot = px.bar(
    execution_time_df, 
    x='database_type', 
    y='total_execution_time',
    title='Total Execution Time in ms (lower is better)',
    template=template,
    height=600,
    width=800
)

total_execution_plot.update_traces(
    marker=
        {'color':'cadetblue'}
    )

# Add a line trace to the existing figure
total_execution_plot.add_trace(go.Scatter(
    x=execution_time_df['database_type'],  
    y=execution_time_df['median_execution_time_ms'],
    mode='lines+markers',
    name='Median Execution Time',  
    marker={'color': 'orange'}  
))



# update layout
total_execution_plot.update_layout(
    title='Total and Median Execution Time (Lower is Better)',
    xaxis_title='Database Type',
    yaxis_title='Execution Time (ms)',
    yaxis_type='log'
)
total_execution_plot.show()


In [122]:
execution_time_box = px.box(
    results_df,
    x='database_type',
    y='execution_time_ms',
    title='Distribution of execution time per database',
    #labels={'database_type': 'Database System', 'cpu_usage_percent': 'Average CPU Utilization (%)'},
    template=template,
    height=600,
    width=800
)

execution_time_box.update_traces(
    marker=
        {'color':'cadetblue'}
    )

execution_time_box.show()

## Cpu usage

In [ ]:
# cpu usage in percent
cpu_usage = duckdb.sql(
    """SELECT 
    database_type,
    ROUND(AVG(cpu_usage_percent), 2) as avg_cpu_percent,
    ROUND(MEDIAN(cpu_usage_percent), 2) as median_cpu_usage_percent,
    ROUND(MIN(cpu_usage_percent), 2) as min_cpu_usage_percent, 
    ROUND(MAX(cpu_usage_percent), 2) as max_cpu_usage_percent,
    ROUND(SUM(cpu_usage_percent), 2) as total_cpu_usage
    FROM results_df
    GROUP BY database_type
    ORDER BY avg_cpu_percent
    """
    )

cpu_usage

┌───────────────────┬─────────────────┬──────────────────────────┬───────────────────────┬───────────────────────┬─────────────────┐
│   database_type   │ avg_cpu_percent │ median_cpu_usage_percent │ min_cpu_usage_percent │ max_cpu_usage_percent │ total_cpu_usage │
│      varchar      │     double      │          double          │        double         │        double         │     double      │
├───────────────────┼─────────────────┼──────────────────────────┼───────────────────────┼───────────────────────┼─────────────────┤
│ DuckDBHandler     │            0.01 │                     0.01 │                   0.0 │                  0.01 │             0.1 │
│ PostgresHandler   │            0.08 │                     0.07 │                  0.04 │                  0.14 │            0.88 │
│ MySQLHandler      │            1.77 │                     1.32 │                  1.06 │                   5.2 │           19.47 │
│ ClickHouseHandler │           12.34 │                     4.54 │   

In [120]:
# create a ploty viz 
cpu_usage_plot = px.bar(
    cpu_usage, 
    x='database_type', 
    y='median_cpu_usage_percent',
    title='Median CPU usage in % (lower is better)',
    template=template,
    height=600,
    width=800
)

cpu_usage_plot.update_traces(
    marker=
        {'color':'lightblue'}
    
    )
cpu_usage_plot.show()

### Memory usage

analyse the min, max, avg, median memory usage in percentage

In [108]:
mem_usage_percent = duckdb.sql(
    """SELECT 
    database_type,
    ROUND(AVG(memory_usage_percent), 2) as avg_memory_usage_percent,
    ROUND(MEDIAN(memory_usage_percent), 2) as median_memory_usage_percent,
    ROUND(MIN(memory_usage_percent), 2) as min_memory_usage_percent, 
    ROUND(MAX(memory_usage_percent), 2) as max_memory_usage_percent
    FROM results_df
    GROUP BY database_type
    ORDER BY avg_memory_usage_percent
    """
    )

mem_usage_percent

┌───────────────────┬──────────────────────────┬─────────────────────────────┬──────────────────────────┬──────────────────────────┐
│   database_type   │ avg_memory_usage_percent │ median_memory_usage_percent │ min_memory_usage_percent │ max_memory_usage_percent │
│      varchar      │          double          │           double            │          double          │          double          │
├───────────────────┼──────────────────────────┼─────────────────────────────┼──────────────────────────┼──────────────────────────┤
│ DuckDBHandler     │                    10.23 │                         8.9 │                      8.9 │                     12.6 │
│ PostgresHandler   │                    13.91 │                       14.22 │                     9.12 │                     17.7 │
│ MySQLHandler      │                    17.53 │                       18.41 │                    13.61 │                    18.65 │
│ ClickHouseHandler │                    18.83 │                     

In [119]:
# create a ploty viz 
mem_usage_plot = px.bar(
    mem_usage_percent, 
    x='database_type', 
    y='median_memory_usage_percent',
    title='Median RAM usage in % (lower is better)',
    template=template,
    height=600,
    width=800
)

mem_usage_plot.update_traces(
    marker=
        {'color':'yellow'}
    
    )
mem_usage_plot.show()

In [112]:
mem_usage_mb = duckdb.sql(
    """SELECT 
    database_type,
    ROUND(AVG(memory_usage_mb), 2) as avg_memory_usage_mb,
    ROUND(MEDIAN(memory_usage_mb), 2) as median_memory_usage_mb,
    ROUND(MIN(memory_usage_mb), 2) as min_memory_usage_mb, 
    ROUND(MAX(memory_usage_mb), 2) as max_memory_usage_mb,
    ROUND(SUM(memory_usage_mb), 2) as total_memory_usage_mb
    FROM results_df
    GROUP BY database_type
    ORDER BY avg_memory_usage_mb
    """
    )

mem_usage_mb

┌───────────────────┬─────────────────────┬────────────────────────┬─────────────────────┬─────────────────────┬───────────────────────┐
│   database_type   │ avg_memory_usage_mb │ median_memory_usage_mb │ min_memory_usage_mb │ max_memory_usage_mb │ total_memory_usage_mb │
│      varchar      │       double        │         double         │       double        │       double        │        double         │
├───────────────────┼─────────────────────┼────────────────────────┼─────────────────────┼─────────────────────┼───────────────────────┤
│ DuckDBHandler     │              104.79 │                  91.13 │               91.13 │              129.04 │               1152.66 │
│ PostgresHandler   │              284.97 │                 291.13 │              186.71 │              362.54 │                3134.7 │
│ MySQLHandler      │              717.84 │                  753.9 │              557.36 │              763.87 │               7896.26 │
│ ClickHouseHandler │              771.27

In [115]:

mem_usage_mb_df = mem_usage_mb.df()

total_memory_mb_plot = px.bar(
    mem_usage_mb_df, 
    x='database_type', 
    y='total_memory_usage_mb',
    title='Total memory usage in MB (lower is better)',
    template=template,
    height=800,
    width=1200
)

total_memory_mb_plot.update_traces(
    marker=
        {'color':'darkred'}
    )

# Add a line trace to the existing figure
total_memory_mb_plot.add_trace(go.Scatter(
    x=mem_usage_mb_df['database_type'],  
    y=mem_usage_mb_df['median_memory_usage_mb'],
    mode='lines+markers',
    name='Median memory usage in MB',  
    marker={'color': 'yellow'}  
))



# update layout
total_memory_mb_plot.update_layout(
    title='Total and Median memory usage in MB (Lower is Better)',
    xaxis_title='Database Type',
    yaxis_title='Memory usage in MB',
    yaxis_type='log'
)
total_memory_mb_plot.show()

### Scatter plot 
Compare the memory usafe (MB) with the execution time (ms)

In [126]:
results_df.columns

Index(['query', 'original_query', 'database_type', 'execution_time_ms',
       'cpu_usage_percent', 'memory_usage_mb', 'memory_usage_percent',
       'disk_read_mb', 'disk_write_mb', 'network_in_mb', 'network_out_mb',
       'result_rows', 'result_size_mb', 'failed'],
      dtype='object')

In [133]:
mem_execution = px.scatter(
   results_df,
   x='execution_time_ms',
   y='memory_usage_mb',
   color='database_type',
   size='memory_usage_percent',
   title='Memory usage vs execution time',
   template=template,
   height=600,
   width=800
)
mem_execution.show()

NB: size represents the percentage of memory usage

## Query execution time analysis

In [24]:
results_df.columns

Index(['query', 'original_query', 'database_type', 'execution_time_ms',
       'cpu_usage_percent', 'memory_usage_mb', 'memory_usage_percent',
       'disk_read_mb', 'disk_write_mb', 'network_in_mb', 'network_out_mb',
       'result_rows', 'result_size_mb', 'failed'],
      dtype='object')

In [25]:
query_time = duckdb.sql(
    """
    SELECT 
        ROUND(MEDIAN(execution_time_ms),1) as median_execution_time_ms,
        original_query as query
    FROM results_df
    GROUP BY 2
    ORDER BY 1 DESC
    LIMIT 10;
    """
)
query_time  

┌──────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ median_execution_time_ms │                                                                                                                                                                              query                                                                                                                                                                              │
│          double          │                                                                                                                                                                             varchar                          

In [26]:
query_time = duckdb.sql(
    """
    WITH parted_table AS (
        SELECT
            database_type,
            execution_time_ms,
            ROW_NUMBER() OVER (PARTITION BY database_type ORDER BY execution_time_ms) as row_num,
            original_query AS query
        FROM
            results_df
        ORDER BY execution_time_ms DESC
        )
    SELECT *
    FROM parted_table
    WHERE row_num <=5
    ORDER BY row_num 
    ;
    """
    )
query_time  

┌───────────────────┬────────────────────┬─────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│   database_type   │ execution_time_ms  │ row_num │                                                                                                                                                                              query                                                                                                                                                                              │
│      varchar      │       double       │  int64  │                                                                                                                                      

### Analyse network usage per database



In [27]:
results_df.columns


Index(['query', 'original_query', 'database_type', 'execution_time_ms',
       'cpu_usage_percent', 'memory_usage_mb', 'memory_usage_percent',
       'disk_read_mb', 'disk_write_mb', 'network_in_mb', 'network_out_mb',
       'result_rows', 'result_size_mb', 'failed'],
      dtype='object')

In [28]:
results_df.describe()

,execution_time_ms,cpu_usage_percent,memory_usage_mb,memory_usage_percent,disk_read_mb,disk_write_mb,network_in_mb,network_out_mb,result_rows,result_size_mb
count,44.000000,44.000000,44.000000,44.000000,44.0,44.0,44.000000,44.000000,44.000000,44.000000
mean,280.579908,3.548964,469.718306,15.125782,0.0,0.0,0.008700,1.361147,42816.818182,1.236386
std,424.818061,12.430645,291.635162,3.919215,0.0,0.0,0.018827,2.849215,72302.088789,2.008356
min,3.624916,0.002186,91.125000,8.898926,0.0,0.0,0.000000,0.000000,0.000000,0.000000
25%,38.366973,0.035220,172.292969,12.556839,0.0,0.0,0.001117,0.001268,1.000000,0.000149
50%,125.228047,0.599989,459.949219,15.779781,0.0,0.0,0.002076,0.004548,100.000000,0.006992
75%,214.842975,3.811686,749.439453,18.296862,0.0,0.0,0.003363,0.369423,48204.000000,1.404179
max,1680.254936,82.681051,910.074219,22.218609,0.0,0.0,0.077224,9.390633,166536.000000,5.082401


In [29]:
network = duckdb.sql("""
    SELECT 
        database_type,
        ROUND(AVG(network_in_mb),2) as avg_network_in_mb, 
        ROUND(AVG(network_out_mb),2) as avg_network_out_mb
    FROM results_df
    GROUP by 1
    """
    )
network

┌───────────────────┬───────────────────┬────────────────────┐
│   database_type   │ avg_network_in_mb │ avg_network_out_mb │
│      varchar      │      double       │       double       │
├───────────────────┼───────────────────┼────────────────────┤
│ MySQLHandler      │              0.01 │               1.83 │
│ ClickHouseHandler │               0.0 │               1.13 │
│ PostgresHandler   │              0.02 │               2.49 │
│ DuckDBHandler     │               0.0 │                0.0 │
└───────────────────┴───────────────────┴────────────────────┘

In [ ]:
# ad check 3d plots